In [1]:
"""
Seed-averaged directional phase scan for the dilute signed long-range Ising
model with three-state couplings J_r in { 0, +|J0|/r^a, -|J0|/r^a }.

Builds on `dilute_signed_lr_ising.py` and uses the same directional
transfer-matrix sink classifier with early stopping
(`determine_phase_at_directional_early_stop`).

For each (p, T) point we run N seeds, count the four possible outcomes
{ferro, antiferro, disorder, undetermined}, and report:

  - the per-seed counts (full information),
  - the majority phase among the *determined* outcomes
    (undetermined seeds are excluded from the denominator),
  - a confidence score = (majority count) / (number of determined seeds).

The plot then renders each point with its majority phase's color, with
saturation determined by confidence:

  confidence >= conf_full     -> full color
  conf_min <= confidence < conf_full
                              -> linearly desaturated toward white
  confidence <  conf_min      -> light gray (ambiguous)
  all seeds undetermined      -> orange (matches single-seed convention)

This keeps a clean, publication-style colored phase diagram while honestly
showing where the method is uncertain.
"""

import numpy as np
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'lib')))
from parallel import *

In [ ]:

# ---- parameters ----
a               = 1.0
r_frac          = 0.3
p_values        = np.linspace(0.0, 1.0, 26)
T_values        = np.linspace(0.01, 3.0, 26)
max_dist_final  = 20
n_steps_total   = 10

base_seed       = 19
n_seeds         = 25      # number of disorder realizations per (p, T)
TM_rs           = (2,)
min_check_step  = 3
thresh_one      = 0.9
thresh_zero     = 0.1
conf_full       = 0.8     # full color above this confidence
conf_min        = 0.6     # ambiguous below this confidence
n_jobs          = -1      # -1 = all cores, 1 = serial, k>1 = k workers

print(
    f"Seed-averaged directional (p, T) phase diagram "
    f"at a={a}, r_frac={r_frac}"
)
print(f"  classification: successive TMs at r={TM_rs}")
print(f"  thresholds: one>{thresh_one}, zero<{thresh_zero}")
print(f"  n_seeds per point: {n_seeds}  (base_seed={base_seed})")
print(f"  confidence shading: full >= {conf_full}, ambiguous < {conf_min}")
print(f"  parallelism: n_jobs={n_jobs}")
print(
    f"  grid: |p|={len(p_values)} x |T|={len(T_values)} = "
    f"{len(p_values) * len(T_values)} points  "
    f"-> {len(p_values) * len(T_values) * n_seeds} RG runs total"
)

point_results, bins = scan_phase_sinks_p_T_directional_early_stop_seed_avg(
    p_values=p_values,
    T_values=T_values,
    a=a,
    r_frac=r_frac,
    max_dist_final=max_dist_final,
    n_steps_total=n_steps_total,
    n_seeds=n_seeds,
    base_seed=base_seed,
    TM_rs=TM_rs,
    min_check_step=min_check_step,
    thresh_one=thresh_one,
    thresh_zero=thresh_zero,
    n_jobs=n_jobs,
    progress="tqdm",
)

# Majority-phase tallies
print(f"  majority ferro:        {len(bins['ferro'])}")
print(f"  majority antiferro:    {len(bins['antiferro'])}")
print(f"  majority disorder:     {len(bins['disorder'])}")
print(f"  all-undetermined:      {len(bins['undetermined'])}")

# Confidence summary
confs = np.array([
    s["confidence"] for s in point_results.values()
    if s["majority_phase"] != "undetermined"
])
if confs.size:
    print(
        f"  confidence (over non-undetermined points): "
        f"mean={confs.mean():.3f}, median={np.median(confs):.3f}, "
        f"min={confs.min():.3f}"
    )
    n_ambig = int((confs < conf_min).sum())
    n_partial = int(((confs >= conf_min) & (confs < conf_full)).sum())
    n_full = int((confs >= conf_full).sum())
    print(
        f"  shading: full={n_full}, partial={n_partial}, "
        f"ambiguous={n_ambig}"
    )

plot_phase_points_p_T_seed_avg(
    point_results=point_results,
    a=a,
    r_frac=r_frac,
    conf_full=conf_full,
    conf_min=conf_min,
)

Seed-averaged directional (p, T) phase diagram at a=1.0, r_frac=0.3
  classification: successive TMs at r=(2,)
  thresholds: one>0.9, zero<0.1
  n_seeds per point: 25  (base_seed=19)
  confidence shading: full >= 0.8, ambiguous < 0.6
  parallelism: n_jobs=-1
  grid: |p|=26 x |T|=26 = 676 points  -> 16900 RG runs total


seed-avg scan a=1, r_frac=0.3, n_seeds=25, n_jobs=-1:   0%|          | 0/16900 [00:00<?, ?it/s]/Users/artun/Documents - Local/GitHub/long-range-ising-chain/lib/parallel.py:234: RuntimeWarning: overflow encountered in exp
  [[np.exp(Jr),  np.exp(-Jr)],
/Users/artun/Documents - Local/GitHub/long-range-ising-chain/lib/parallel.py:235: RuntimeWarning: overflow encountered in exp
  [np.exp(-Jr), np.exp(Jr)]],
/Users/artun/Documents - Local/GitHub/long-range-ising-chain/lib/parallel.py:239: RuntimeWarning: invalid value encountered in divide
  T = T / T.max()
seed-avg scan a=1, r_frac=0.3, n_seeds=25, n_jobs=-1:   0%|          | 1/16900 [00:06<30:04:08,  6.41s/it]/Users/artun/Documents - Local/GitHub/long-range-ising-chain/lib/parallel.py:234: RuntimeWarning: overflow encountered in exp
  [[np.exp(Jr),  np.exp(-Jr)],
/Users/artun/Documents - Local/GitHub/long-range-ising-chain/lib/parallel.py:235: RuntimeWarning: overflow encountered in exp
  [np.exp(-Jr), np.exp(Jr)]],
/Users/artun/Document